# TribonacciDNLS — Annotated Notebook

**Paper:** Differential Nonlinear Robustness of Critical States in Fibonacci and Tribonacci Substitution Chains
Pablo Nogueira Grossi · G6 LLC · Newark NJ · 2026
DOI: 10.5281/zenodo.20075822 (v4 draft)

**Pipeline:** substitution chain generation -> Hamiltonian -> mid-gap eigenstate -> IPR -> DNLS evolution -> fig1-9

**Dependencies:** numpy, scipy, matplotlib

In [ ]:
# Cell 1 - Imports & style
import numpy as np
from scipy.linalg import eigh
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Georgia'],
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.05,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.8, 'lines.linewidth': 1.8,
    'grid.alpha': 0.3, 'grid.linewidth': 0.5
})

COL_FIB  = '#2166ac'
COL_TRIB = '#d6604d'
COL_GOLD = '#c9a84c'
COL_GREY = '#888888'
ETA = 1.8392867552141612
PHI = 1.6180339887498949

In [ ]:
# Cell 2 - Substitution chain generation
# Fibonacci:  A->AB, B->A         (letters 0,1)
# Tribonacci: 1->12, 2->13, 3->1  (letters 0,1,2)
# Companion matrix T3 in SL(3,Z), spectral radius = eta

def fibonacci_word(length):
    word = [0]
    rules = {0: [0, 1], 1: [0]}
    while len(word) < length:
        word = [s for c in word for s in rules[c]]
    return word[:length]

def tribonacci_word(length):
    word = [0]
    rules = {0: [0, 1], 1: [0, 2], 2: [0]}
    while len(word) < length:
        word = [s for c in word for s in rules[c]]
    return word[:length]

wt = tribonacci_word(10000)
print('Tribonacci freqs:', round(wt.count(0)/10000,4),
      round(wt.count(1)/10000,4), round(wt.count(2)/10000,4))
print('Expected:         0.5437             0.2956             0.1607')

In [ ]:
# Cell 3 - Tight-binding Hamiltonian & mid-gap eigenstate
# H[j,j+1] = H[j+1,j] = t_word[j], diagonal = 0
# letter 0->1.0, letter 1->t_mod, letter 2->t_mod^2
# t_mod=0.5 (paper default)

T_MOD = 0.5
N = 500

def build_hamiltonian(word, N, t_mod=T_MOD):
    hop_map  = {0: 1.0, 1: t_mod, 2: t_mod**2}
    hoppings = np.array([hop_map.get(word[j], t_mod) for j in range(N-1)])
    H = np.zeros((N, N))
    for j in range(N-1):
        H[j, j+1] = hoppings[j]
        H[j+1, j] = hoppings[j]
    return H, hoppings

def mid_gap_state(H):
    vals, vecs = eigh(H)
    idx = np.argmin(np.abs(vals))
    return vecs[:, idx], vals[idx]

word_fib  = fibonacci_word(N+1)
word_trib = tribonacci_word(N+1)
H_fib,  hop_fib  = build_hamiltonian(word_fib,  N)
H_trib, hop_trib = build_hamiltonian(word_trib, N)
psi0_fib,  E_fib  = mid_gap_state(H_fib)
psi0_trib, E_trib = mid_gap_state(H_trib)
print(f'Fibonacci  E_mid = {E_fib:.6f}')
print(f'Tribonacci E_mid = {E_trib:.6f}')

In [ ]:
# Cell 4 - IPR = sum|psi|^4 / (sum|psi|^2)^2

def ipr(psi):
    norm2 = np.sum(np.abs(psi)**2)
    return np.sum(np.abs(psi)**4) / norm2**2

ipr0_fib  = ipr(psi0_fib)
ipr0_trib = ipr(psi0_trib)
print(f'IPR(lam=0) Fibonacci : {ipr0_fib:.6f}')
print(f'IPR(lam=0) Tribonacci: {ipr0_trib:.6f}')
print(f'Ratio trib/fib       : {ipr0_trib/ipr0_fib:.4f}')

In [ ]:
# Cell 5 - DNLS evolution
# i dpsi/dt = -H psi + lam|psi|^2 psi
# Real split: state=[Re(psi); Im(psi)]
# dx/dt = -Hy + lam|psi|^2 y
# dy/dt =  Hx - lam|psi|^2 x
# Use integrator='DOP853' for T >= 1e4

def evolve_dnls(psi0, hoppings, lam, T_evo,
                integrator='RK45', rtol=1e-8, atol=1e-10):
    N  = len(psi0)
    h  = hoppings
    s0 = np.concatenate([psi0.real, psi0.imag])

    def rhs(t, state):
        x, y = state[:N], state[N:]
        Hx = np.zeros(N); Hy = np.zeros(N)
        Hx[:-1] += h*x[1:];  Hx[1:] += h*x[:-1]
        Hy[:-1] += h*y[1:];  Hy[1:] += h*y[:-1]
        mod2 = x**2 + y**2
        return np.concatenate([-Hy + lam*mod2*y,
                                 Hx - lam*mod2*x])

    t_eval = np.unique(np.concatenate(
        [[0.0], np.logspace(0, np.log10(T_evo), 200)]))
    sol = solve_ivp(rhs, [0, T_evo], s0, method=integrator,
                    t_eval=t_eval, rtol=rtol, atol=atol)
    psi_f = sol.y[:N,-1] + 1j*sol.y[N:,-1]
    drift = abs(np.sum(np.abs(psi_f)**2) - np.sum(np.abs(psi0)**2))
    if drift / np.sum(np.abs(psi0)**2) > 1e-4:
        print(f'WARNING norm drift {drift:.2e} lam={lam} T={T_evo}')
    return psi_f, sol.t

pf_t, _ = evolve_dnls(psi0_fib,  hop_fib,  1.5, 50.0)
pt_t, _ = evolve_dnls(psi0_trib, hop_trib, 1.5, 50.0)
print(f'Fib  drop: {100*(ipr0_fib -ipr(pf_t))/ipr0_fib :.1f}%  (expect ~57%)')
print(f'Trib drop: {100*(ipr0_trib-ipr(pt_t))/ipr0_trib:.1f}%  (expect <5%)')

In [ ]:
# Cell 6 - Full lambda scan -> Table 1

T_EVO  = 50.0
LAMBDA = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 7.0, 10.0]

ipr_fib_list  = [ipr0_fib]
ipr_trib_list = [ipr0_trib]

for lam in LAMBDA[1:]:
    pf, _ = evolve_dnls(psi0_fib,  hop_fib,  lam, T_EVO)
    pt, _ = evolve_dnls(psi0_trib, hop_trib, lam, T_EVO)
    ipr_fib_list.append(ipr(pf))
    ipr_trib_list.append(ipr(pt))

ipr_fib_arr  = np.array(ipr_fib_list)
ipr_trib_arr = np.array(ipr_trib_list)
ratio_arr    = ipr_trib_arr / ipr_fib_arr
lam_arr      = np.array(LAMBDA)

print(f"{'lam':>5} {'IPR_trib':>10} {'IPR_fib':>10} {'ratio':>7} {'dtrib%':>8} {'dfib%':>7}")
for i, lam in enumerate(LAMBDA):
    dt = 100*(ipr0_trib-ipr_trib_arr[i])/ipr0_trib
    df = 100*(ipr0_fib -ipr_fib_arr[i]) /ipr0_fib
    print(f'{lam:>5.1f} {ipr_trib_arr[i]:>10.4f} {ipr_fib_arr[i]:>10.4f} '
          f'{ratio_arr[i]:>7.3f} {dt:>7.1f}% {df:>7.1f}%')

In [ ]:
# Cell 7 - Fig 1: Chain structure diagram

def make_chain_diagram(ax, word, hops, n_show=30, label='',
                       col_A='#2166ac', col_B='#d6604d', col_C='#4dac26'):
    lc = {0: col_A, 1: col_B, 2: col_C}
    ls = {0: 'A', 1: 'B', 2: 'C'}
    ax.set_xlim(-0.5, n_show-0.5); ax.set_ylim(-0.6, 0.8); ax.axis('off')
    for j in range(n_show):
        ax.add_patch(mpatches.FancyBboxPatch(
            (j-0.35,-0.2), 0.7, 0.4, boxstyle='round,pad=0.05',
            facecolor=lc.get(word[j],'#888'), edgecolor='white',
            linewidth=0.5, zorder=2))
        ax.text(j, 0.0, ls.get(word[j],'?'), ha='center', va='center',
                fontsize=7, color='white', fontweight='bold', zorder=3)
    for j in range(n_show-1):
        t = hops[j]
        ax.plot([j+0.35,j+0.65],[0,0], color='#444',
                lw=1.0+3.0*t, alpha=0.4+0.5*t, zorder=1)
    ax.set_title(label, fontsize=10, pad=4)

fig1, ax1 = plt.subplots(2, 1, figsize=(12, 3.5))
fig1.suptitle('Chain structure: first 30 sites', fontsize=12, y=1.02)
make_chain_diagram(ax1[0], word_fib,  hop_fib,
                   label=f'Fibonacci phi={PHI:.3f}',
                   col_A=COL_FIB, col_B='#74add1')
make_chain_diagram(ax1[1], word_trib, hop_trib,
                   label=f'Tribonacci eta={ETA:.3f}',
                   col_A=COL_TRIB, col_B='#f4a582', col_C='#4dac26')
fig1.legend(handles=[
    mpatches.Patch(facecolor=COL_FIB,   label='Fib A t=1.0'),
    mpatches.Patch(facecolor='#74add1', label='Fib B t=0.5'),
    mpatches.Patch(facecolor=COL_TRIB,  label='Tri A t=1.0'),
    mpatches.Patch(facecolor='#f4a582', label='Tri B t=0.5'),
    mpatches.Patch(facecolor='#4dac26', label='Tri C t=0.25'),
], loc='lower center', ncol=5, bbox_to_anchor=(0.5,-0.12), fontsize=8)
plt.tight_layout()
fig1.savefig('fig1_chain_structure.pdf')
fig1.savefig('fig1_chain_structure.png')
plt.show(); print('fig1 done')

In [ ]:
# Cell 8 - Fig 2: Mid-gap eigenstates |psi|^2

fig2, ax2 = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
fig2.suptitle('Mid-gap eigenstates |psi_j|^2  (lambda=0)', fontsize=12)
sites = np.arange(N)
for ax, psi, col, lbl, iv, ev in [
    (ax2[0], psi0_fib,  COL_FIB,  f'Fibonacci  phi={PHI:.3f}',  ipr0_fib,  E_fib),
    (ax2[1], psi0_trib, COL_TRIB, f'Tribonacci eta={ETA:.3f}', ipr0_trib, E_trib)
]:
    ax.bar(sites, np.abs(psi)**2, width=1.0, color=col, alpha=0.7)
    ax.set_ylabel('|psi_j|^2')
    ax.set_title(f'{lbl}   E={ev:.4f}   IPR={iv:.4f}', fontsize=10)
    ax.set_xlim(0, N); ax.grid(True, axis='y', alpha=0.3)
ax2[1].set_xlabel('Site j')
plt.tight_layout()
fig2.savefig('fig2_eigenstates.pdf')
fig2.savefig('fig2_eigenstates.png')
plt.show(); print('fig2 done')

In [ ]:
# Cell 9 - Fig 3: IPR vs lambda (MAIN RESULT)
# At lam=1.5: Fibonacci drops ~57%, Tribonacci <5%

fig3, ax3 = plt.subplots(figsize=(7, 4.5))
ax3.plot(lam_arr, ipr_trib_arr, 'o-', color=COL_TRIB, lw=2, ms=6,
         label=f'Tribonacci eta={ETA:.3f}')
ax3.plot(lam_arr, ipr_fib_arr,  's-', color=COL_FIB,  lw=2, ms=6,
         label=f'Fibonacci  phi={PHI:.3f}')
idx15 = LAMBDA.index(1.5)
dt15  = 100*(ipr0_trib-ipr_trib_arr[idx15])/ipr0_trib
df15  = 100*(ipr0_fib -ipr_fib_arr[idx15]) /ipr0_fib
ax3.annotate(f'trib drop {dt15:.0f}%',
             xy=(1.5, ipr_trib_arr[idx15]),
             xytext=(2.0, ipr_trib_arr[idx15]+0.003), fontsize=8, color=COL_TRIB,
             arrowprops=dict(arrowstyle='->', color=COL_TRIB, lw=0.8))
ax3.annotate(f'fib drop {df15:.0f}%',
             xy=(1.5, ipr_fib_arr[idx15]),
             xytext=(2.0, ipr_fib_arr[idx15]-0.004), fontsize=8, color=COL_FIB,
             arrowprops=dict(arrowstyle='->', color=COL_FIB, lw=0.8))
ax3.axvline(1.5, color=COL_GREY, lw=0.8, ls='--', alpha=0.6)
ax3.set_xlabel('Nonlinearity lambda')
ax3.set_ylabel('IPR mid-gap state')
ax3.set_title(f'Differential Nonlinear Robustness  N={N} T={T_EVO}')
ax3.legend(); ax3.grid(True, alpha=0.3)
plt.tight_layout()
fig3.savefig('fig3_ipr_vs_lambda.pdf')
fig3.savefig('fig3_ipr_vs_lambda.png')
plt.show(); print('fig3 done')

In [ ]:
# Cell 10 - Fig 4: Trib/Fib IPR ratio vs lambda

fig4, ax4 = plt.subplots(figsize=(7, 4))
ax4.plot(lam_arr, ratio_arr, 'D-', color=COL_GOLD, lw=2.2, ms=6,
         markeredgecolor='#8a6a00', markeredgewidth=0.5, label='IPR_trib/IPR_fib')
ax4.axhline(1.0, color=COL_GREY, lw=0.8, ls='--', label='ratio=1')
ax4.axhline(4.0, color=COL_GREY, lw=0.5, ls=':', alpha=0.6, label='4x ref')
pk = np.argmax(ratio_arr)
ax4.annotate(f'{ratio_arr[pk]:.2f}x  lam={lam_arr[pk]:.1f}',
             xy=(lam_arr[pk], ratio_arr[pk]),
             xytext=(lam_arr[pk]+0.5, ratio_arr[pk]-0.3), fontsize=8, color=COL_GOLD,
             arrowprops=dict(arrowstyle='->', color=COL_GOLD, lw=0.8))
ax4.set_xlabel('Nonlinearity lambda')
ax4.set_ylabel('IPR ratio trib/fib')
ax4.set_title(f'IPR ratio vs lambda  N={N} T={T_EVO}')
ax4.legend(); ax4.grid(True, alpha=0.3)
plt.tight_layout()
fig4.savefig('fig4_ipr_ratio.pdf')
fig4.savefig('fig4_ipr_ratio.png')
plt.show(); print('fig4 done')

In [ ]:
# Cell 11 - Fig 5: Substitution tree

def draw_tree(ax, rules, names, colors, n_gen=4, x0=0.5, y0=0.92, dx=0.42):
    def node(letter, x, y, depth, parent=None):
        if parent:
            ax.annotate('', xy=(x,y+0.02), xytext=parent,
                        arrowprops=dict(arrowstyle='->', lw=0.7, color='#666'))
        ax.scatter([x],[y], s=180, color=colors[letter], zorder=3,
                   edgecolors='white', linewidth=0.8)
        ax.text(x, y, names[letter], ha='center', va='center',
                fontsize=7, color='white', fontweight='bold', zorder=4)
        if depth >= n_gen: return
        ch = rules[letter]
        xs = np.linspace(x-dx/2**depth, x+dx/2**depth, len(ch))
        for i,c in enumerate(ch):
            node(c, xs[i], y-0.16, depth+1, (x,y-0.02))
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
    node(0, x0, y0, 0)

fig5, ax5 = plt.subplots(1, 2, figsize=(12, 5))
fig5.suptitle('Substitution trees (4 generations)', fontsize=12)
draw_tree(ax5[0],
    rules={0:[0,1], 1:[0]},
    names={0:'A', 1:'B'},
    colors={0:COL_FIB, 1:'#74add1'})
ax5[0].set_title('Fibonacci: A->AB, B->A', fontsize=10)
draw_tree(ax5[1],
    rules={0:[0,1], 1:[0,2], 2:[0]},
    names={0:'1', 1:'2', 2:'3'},
    colors={0:COL_TRIB, 1:'#f4a582', 2:'#4dac26'},
    n_gen=3)
ax5[1].set_title('Tribonacci: 1->12, 2->13, 3->1', fontsize=10)
plt.tight_layout()
fig5.savefig('fig5_substitution_tree.pdf')
fig5.savefig('fig5_substitution_tree.png')
plt.show(); print('fig5 done')

In [ ]:
# Cell 13 - Fig 7: Finite-size scaling at T=1e4
# Data from Brief 2 (embedded)
# Grouped bars: trib/fib ratio at each (lambda, N)

Ns      = [500, 1000, 2000]
lam_fss = [0.5, 1.0, 1.5, 2.0]
ratio_fss = [
    [1.08, 1.12, 1.15],
    [1.41, 1.68, 1.82],
    [4.18, 3.21, 3.89],
    [2.31, 2.74, 3.12]
]

fig7, ax7 = plt.subplots(figsize=(9, 4.5))
x     = np.arange(len(lam_fss))
width = 0.22
bar_colors = ['#4393c3', '#2166ac', '#053061']
for i, (N_val, col) in enumerate(zip(Ns, bar_colors)):
    vals = [ratio_fss[j][i] for j in range(len(lam_fss))]
    ax7.bar(x + (i-1)*width, vals, width, color=col,
            alpha=0.85, label=f'N={N_val}')
ax7.axhline(1.0, color=COL_GREY, lw=0.8, ls='--')
ax7.set_xticks(x)
ax7.set_xticklabels([f'lam={l}' for l in lam_fss])
ax7.set_ylabel('IPR ratio trib/fib')
ax7.set_title('Finite-size scaling T=1e4  (Brief 2 data)', fontsize=11)
ax7.legend(fontsize=9)
ax7.annotate('non-monotone\nlam=1.5 row\n(resolved at T=1e5)',
             xy=(2, 4.18), xytext=(2.4, 4.4), fontsize=7, color=COL_GREY,
             arrowprops=dict(arrowstyle='->', color=COL_GREY, lw=0.7))
ax7.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
fig7.savefig('fig7_fss_T1e4.pdf')
fig7.savefig('fig7_fss_T1e4.png')
plt.show(); print('fig7 done')

In [ ]:
# Cell 14 - Fig 8: Ratio collapse T=1e4 -> T=1e5
# Data from Brief 4 (embedded)
# Bar pairs at N=500, 1000, 2000 showing collapse from T=1e4 to T=1e5

Ns_coll   = [500, 1000, 2000]
ratio_T4  = [3.95, 4.18, 3.89]
ratio_T5  = [1.18, 1.20, 1.22]

fig8, ax8 = plt.subplots(figsize=(7, 4.5))
x     = np.arange(len(Ns_coll))
width = 0.35
ax8.bar(x - width/2, ratio_T4, width, color='#d6604d', alpha=0.85, label='T=1e4')
ax8.bar(x + width/2, ratio_T5, width, color='#4393c3', alpha=0.85, label='T=1e5')
ax8.axhline(1.0, color=COL_GREY, lw=0.8, ls='--')
ax8.set_xticks(x)
ax8.set_xticklabels([f'N={n}' for n in Ns_coll])
ax8.set_ylabel('IPR ratio trib/fib')
ax8.set_title('Ratio collapse T=1e4 to T=1e5  lam=1.5  (Brief 4 data)', fontsize=11)
ax8.legend(fontsize=9)
ax8.annotate('~5% run-to-run\nuncertainty at T>=1e5',
             xy=(1, 1.20), xytext=(1.5, 1.6), fontsize=7, color=COL_GREY,
             arrowprops=dict(arrowstyle='->', color=COL_GREY, lw=0.7))
ax8.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
fig8.savefig('fig8_ratio_collapse.pdf')
fig8.savefig('fig8_ratio_collapse.png')
plt.show(); print('fig8 done')

In [ ]:
# Cell 15 - Fig 9: Long-time saturation to T=1e6
# Data: Brief 4 direct measurements at T=1e4, 1e5, 3e5, 1e6 (N=1000, lam=1.5)
# Saturation band: ratio = 1.04 +/- 0.04 at T~3e5

T_pts   = np.array([1e4, 1e5, 3e5, 1e6])
rat_pts = np.array([4.18, 1.20, 1.06, 1.04])
rat_lo  = np.array([4.18, 1.16, 1.01, 1.00])
rat_hi  = np.array([4.18, 1.24, 1.10, 1.08])

fig9, ax9 = plt.subplots(figsize=(8, 4.5))
ax9.semilogx(T_pts, rat_pts, 'o-', color=COL_GOLD, lw=2, ms=8,
             markeredgecolor='#8a6a00', markeredgewidth=0.8,
             label='trib/fib IPR ratio  N=1000  lam=1.5')
ax9.fill_between(T_pts, rat_lo, rat_hi, color=COL_GOLD, alpha=0.18,
                 label='run-to-run band (~5%)')
ax9.axhline(1.0, color=COL_GREY, lw=0.8, ls='--', label='ratio=1')
ax9.axhspan(1.00, 1.08, color=COL_GOLD, alpha=0.08)
ax9.annotate('saturation\n1.04 +/- 0.04',
             xy=(3e5, 1.04), xytext=(4e4, 0.82), fontsize=9, color=COL_GOLD,
             arrowprops=dict(arrowstyle='->', color=COL_GOLD, lw=1.0))
ax9.set_xlabel('Evolution time T')
ax9.set_ylabel('IPR ratio trib/fib')
ax9.set_title('Long-time saturation  N=1000  lam=1.5  (Brief 4 data)', fontsize=11)
ax9.legend(fontsize=9)
ax9.grid(True, which='both', alpha=0.3)
ax9.set_ylim(0.7, 4.8)
plt.tight_layout()
fig9.savefig('fig9_T1e6_saturation.pdf')
fig9.savefig('fig9_T1e6_saturation.png')
plt.show(); print('fig9 done')
print('\nAll figures complete.')